# 1 · Data exploration & preprocessing

Corporate credit-rating prediction from 16 financial-ratio features. This
notebook loads the raw data, applies the thesis preprocessing pipeline, and
creates the **company-aware** train/test split that every later experiment reuses.

All heavy lifting lives in the `credit_rating` package — this notebook is a thin,
explained wrapper around it.

In [ ]:
# Make the credit_rating package importable whether or not it's pip-installed.
try:
    import credit_rating  # noqa: F401
except ModuleNotFoundError:
    import sys, pathlib
    sys.path.insert(0, str(pathlib.Path.cwd().parent))

from credit_rating import config, data, models, evaluate, experiments, plots
print("Package loaded. Device:", config.get_device())

## Load the raw data

Place `corporateCreditRatingWithFinancialRatios.csv` in the `data/` folder (or
point `THESIS_DATA_DIR` at it). On Colab you can instead set
`THESIS_DATA_DIR=/content/drive/MyDrive/THESIS/clean_redo`.

In [ ]:
df_raw = data.load_raw()
print("Raw shape:", df_raw.shape)
df_raw[["Rating"] + config.FEATURE_COLS[:4]].head()

## Preprocessing

Three steps, all in `data.preprocess`:

1. **Collapse notched ratings** — `BB+`/`BB-` to `BB`, giving 10 broad classes.
2. **Winsorise** each feature at the 1st/99th percentile to tame outliers.
3. **Encode** ratings as integers `D=0 ... AAA=9` (`Rating_Num`).

In [ ]:
df = data.preprocess(df_raw)
dist = df["Rating"].value_counts().reindex(
    [r for r in config.RATING_LABELS if r in df["Rating"].unique()])
print(dist)

In [ ]:
import matplotlib.pyplot as plt
dist.plot(kind="bar", color="slategray", edgecolor="white", figsize=(9, 4))
plt.title("Class distribution (broad ratings)")
plt.ylabel("Count"); plt.xlabel("Rating"); plt.tight_layout(); plt.show()

## Company-aware split

A `GroupShuffleSplit` on the company column (`config.GROUP_COL`) guarantees no
company appears in both train and test — otherwise the model could memorise a
firm's profile and leak across the split. The split is saved to
`results/splits.pkl` and never regenerated.

In [ ]:
X_train, X_test, y_train, y_test = data.prepare_and_save_splits()
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("Classes in train:", sorted(y_train.unique()))
print("Classes in test :", sorted(y_test.unique()))
print("Saved ->", config.SPLITS_PATH)